# Exercise 4: Language tokens

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Speech Technologies · version 0.1.0*

> Decode Kazakh and Turkish speech with the Whisper tokens of related languages and with automatic language detection. Check which Turkic language tokens the model accepts; for Kyrgyz or Uyghur, compare related-language and multilingual initialisations by fine-tuning instead of forcing an unsupported token.

**Setup.** `pip install sherpa-onnx==1.13.8 librosa==0.11.0 soundfile requests numpy`. Everything runs on the CPU without a GPU or the Hugging Face Hub:

- **ASR:** OpenAI Whisper *small*, as the int8 ONNX export published in the sherpa-onnx GitHub release `asr-models` (639 MB archive), decoded greedily by sherpa-onnx.
- **TTS:** the Piper VITS voice `kk_KZ-issai-high` (six speakers, trained from scratch on the ISSAI KazakhTTS data, CC BY 4.0 according to its model card) and, in Exercise 4, `tr_TR-dfki-medium` (one speaker, CC BY-NC-SA 4.0), from the sherpa-onnx release `tts-models`.
- **Text:** the Kazakh and Turkish sides of the X-WMT English–Kazakh and English–Turkish test sets in the commit-pinned TIL release (MIT licence), with the spaces around punctuation removed.

Archives are checked against the SHA-256 digests in the code and cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), together with every synthesised waveform and every transcript, so a rerun reuses them. **The test speech is synthetic.** It comes from a TTS model trained on read studio speech; its error rates say nothing about Whisper on real Kazakh recordings, and they are used here only to compare conditions on identical input. Piper samples its output (noise scale 0.667), so a fresh cache produces slightly different audio and slightly different numbers.

Whisper's decoder starts with a language token. The first part probes which Turkic tokens the model accepts. Then the same 30 Kazakh utterances from Exercise 1 and 20 Turkish ones (Piper `tr_TR-dfki-medium`) are decoded with the correct token, with tokens of other languages and with automatic detection (`auto`, where the result reports the detected language). *Cyrillic* is the share of Cyrillic letters in the output.

In [ ]:
import collections, hashlib, json, os, re, statistics, tarfile, time, unicodedata
from pathlib import Path
import numpy as np, requests

TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/"
SHERPA = "https://github.com/k2-fsa/sherpa-onnx/releases/download/"
ARCHIVES = {   # sherpa-onnx release archive -> SHA-256 of the file used for the book's run
    "asr-models/sherpa-onnx-whisper-small.tar.bz2": "486a46afbb7ba798507190ffe02fea2dd726049af212e774537efac6afb210a6",
    "tts-models/vits-piper-kk_KZ-issai-high.tar.bz2": "6c59955f7f5e3fc50b130ced59a430eb4ef62d69155af4ce836db0dde4a42682",
    "tts-models/vits-piper-tr_TR-dfki-medium.tar.bz2": "b63babaefcdd63202c7079aba25b72a2013825ad5895e2aeff03d1ae5a5b3bbf",
}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
FETCHED = {}                                   # url -> sha256 of every file read (saved with the outputs)
SR = 16000                                     # Whisper's sampling rate
THREADS = os.cpu_count() or 1

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        with requests.get(url, timeout=600, stream=True) as r:
            r.raise_for_status()
            with open(str(path) + ".part", "wb") as f:
                for block in r.iter_content(1 << 20):
                    f.write(block)
        os.replace(str(path) + ".part", path)
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    FETCHED[url] = h.hexdigest()
    return path

def archive(name):
    """Download, verify and unpack a sherpa-onnx release archive; return the unpacked folder."""
    path = fetch(SHERPA + name)
    if FETCHED[SHERPA + name] != ARCHIVES[name]:
        raise ValueError(f"{name}: the release file has changed; check it before using it")
    root = path.parent / path.name.removesuffix(".tar.bz2")
    if not root.exists():
        with tarfile.open(path) as t:
            t.extractall(path.parent, filter="data")
    return root

def detok(s):
    """Undo the X-WMT tokenisation (spaces around punctuation)."""
    s = re.sub(r" ([’'] ?)", lambda m: m.group(1).strip(), s)
    s = re.sub(r" ([.,!?;:%)\]»])", r"\1", s)
    return re.sub(r"([(\[«]) ", r"\1", s).strip()

def xwmt_side(pair, lang):
    """One side of an X-WMT test set, detokenised."""
    text = fetch(f"{TIL}xwmt/test/{pair}/{pair}.{lang}").read_text(encoding="utf-8").splitlines()
    return [detok(s) for s in text if s.strip()]

def test_sentences(pair, lang, n):
    """The first n sentences with 5-20 words and no digits (number expansion is a separate problem)."""
    return [s for s in xwmt_side(pair, lang) if not re.search(r"\d", s) and 5 <= len(s.split()) <= 20][:n]

# ---- TTS: Piper VITS voices through sherpa-onnx; every waveform is cached on disk -------------
VOICES = {"kk": "kk_KZ-issai-high", "tr": "tr_TR-dfki-medium"}
_TTS = {}

def tts(voice, noise=None):
    if (voice, noise) not in _TTS:
        import sherpa_onnx
        root = archive(f"tts-models/vits-piper-{voice}.tar.bz2")
        vits = sherpa_onnx.OfflineTtsVitsModelConfig(model=str(root / f"{voice}.onnx"), tokens=str(root / "tokens.txt"),
                                                     data_dir=str(root / "espeak-ng-data"))
        if noise is not None:                   # noise=0 makes the voice deterministic
            vits.noise_scale = vits.noise_scale_w = noise
        _TTS[voice, noise] = sherpa_onnx.OfflineTts(sherpa_onnx.OfflineTtsConfig(
            model=sherpa_onnx.OfflineTtsModelConfig(vits=vits, num_threads=THREADS)))
    return _TTS[voice, noise]

def speech(text, sid=0, lang="kk", draw=0, noise=None):
    """16 kHz float32 synthetic speech for text; `draw` indexes repeated samples of the same input."""
    key = hashlib.sha1(json.dumps([VOICES[lang], sid, draw, noise, text], ensure_ascii=False).encode()).hexdigest()
    path = CACHE / "ch11-audio" / f"{key}.npy"
    if not path.exists():
        import librosa
        a = tts(VOICES[lang], noise).generate(text, sid=sid, speed=1.0)
        w = librosa.resample(np.asarray(a.samples, dtype=np.float32), orig_sr=a.sample_rate, target_sr=SR)
        path.parent.mkdir(parents=True, exist_ok=True)
        np.save(path, w.astype(np.float32))
    return np.load(path)

# ---- ASR: Whisper small (int8 ONNX), greedy; every transcript is cached ----------------------
_ASR = {}
ASR_CACHE = CACHE / "ch11-transcripts.jsonl"

def _asr(lang):
    if lang not in _ASR:
        import sherpa_onnx
        root = archive("asr-models/sherpa-onnx-whisper-small.tar.bz2")
        _ASR[lang] = sherpa_onnx.OfflineRecognizer.from_whisper(
            encoder=str(root / "small-encoder.int8.onnx"), decoder=str(root / "small-decoder.int8.onnx"),
            tokens=str(root / "small-tokens.txt"), language=lang, task="transcribe", num_threads=THREADS)
    return _ASR[lang]

def _transcripts():
    out = {}
    if ASR_CACHE.exists():
        for line in ASR_CACHE.read_text(encoding="utf-8").splitlines():
            r = json.loads(line); out[r["key"]] = r
    return out
TRANSCRIPTS = _transcripts()

def transcribe(wave, lang="kk"):
    """(text, language Whisper used, decoding seconds); lang="" lets Whisper detect the language."""
    wave = np.ascontiguousarray(wave, dtype=np.float32)
    key = hashlib.sha1(wave.tobytes()).hexdigest() + "|" + lang
    if key not in TRANSCRIPTS:
        rec = _asr(lang)
        s = rec.create_stream(); s.accept_waveform(SR, wave)
        t0 = time.time(); rec.decode_stream(s)
        r = {"key": key, "text": s.result.text.strip(), "lang": s.result.lang, "sec": time.time() - t0}
        TRANSCRIPTS[key] = r
        ASR_CACHE.parent.mkdir(parents=True, exist_ok=True)
        with ASR_CACHE.open("a", encoding="utf-8") as f:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    r = TRANSCRIPTS[key]
    return r["text"], r["lang"], r["sec"]

# ---- Scoring: the same functions as code/remaining/speech.py -----------------------------------
def normalize(text):
    # Conservative scoring: preserve letters, diacritics and apostrophes.
    text = unicodedata.normalize("NFC", text).lower()
    return " ".join("".join(c if not unicodedata.category(c).startswith("P") or c in "'ʻʼ’" else " " for c in text).split())

def distance(a, b):
    row = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        nxt = [i]
        for j, y in enumerate(b, 1):
            nxt.append(min(nxt[-1] + 1, row[j] + 1, row[j - 1] + (x != y)))
        row = nxt
    return row[-1]

def errors(refs, hyps):
    """Corpus-level raw and normalised WER and CER (edits / reference words or characters)."""
    if len(refs) != len(hyps) or not refs:
        raise ValueError("unaligned/empty references")
    result = {}
    for name, fn in [("raw", lambda x: x), ("normalized", normalize)]:
        pairs = [(fn(r), fn(h)) for r, h in zip(refs, hyps)]
        for unit, split in [("wer", str.split), ("cer", list)]:
            denom = sum(len(split(r)) for r, h in pairs)
            edits = sum(distance(split(r), split(h)) for r, h in pairs)
            result[f"{name}_{unit}"] = edits / denom if denom else None
        result[f"{name}_empty_references"] = sum(not r for r, h in pairs)
    return result

def pct(x):
    return f"{100 * x:5.1f}%"

def audio_digest(waves):
    h = hashlib.sha256()
    for w in waves:
        h.update(np.ascontiguousarray(w, dtype=np.float32).tobytes())
    return h.hexdigest()[:16]

print("archives verified:", ", ".join(n.rsplit("/", 1)[-1] for n in ARCHIVES if archive(n)))

In [ ]:
kk_all = xwmt_side("en-kk", "kk")
TEST = test_sentences("en-kk", "kk", 60)               # 60 sentences, spoken by the six voices in turn
waves = [speech(s, sid=i % 6) for i, s in enumerate(TEST)]
hyps, langs, secs = zip(*(transcribe(w, "kk") for w in waves))
audio = sum(len(w) for w in waves) / SR
print(f"X-WMT en-kk test: {len(kk_all)} Kazakh sentences; the first {len(TEST)} with 5-20 words and no digits are used")
print(f"synthetic audio: {audio / 60:.1f} min, {min(len(w) for w in waves) / SR:.1f}-{max(len(w) for w in waves) / SR:.1f} s per item; audio digest {audio_digest(waves)}")
print(f"decoding: {sum(secs):.0f} s on {THREADS} CPU threads, real-time factor {sum(secs) / audio:.2f}")
r = errors(TEST, list(hyps))
print(f"Whisper small, language token kk, greedy:  raw WER {pct(r['raw_wer'])}  CER {pct(r['raw_cer'])}   "
      f"normalised WER {pct(r['normalized_wer'])}  CER {pct(r['normalized_cer'])}")
short = sum(len(normalize(h)) < 0.5 * len(normalize(r)) for r, h in zip(TEST, hyps))
print(f"hypotheses shorter than half their reference (normalised characters): {short} of {len(TEST)}")
print("normalised CER by voice:")
for sid in range(6):
    idx = [i for i in range(len(TEST)) if i % 6 == sid]
    e = errors([TEST[i] for i in idx], [hyps[i] for i in idx])
    print(f"  speaker {sid}: CER {pct(e['normalized_cer'])}  WER {pct(e['normalized_wer'])}  ({len(idx)} items)")
item = sorted(range(len(TEST)), key=lambda i: distance(list(normalize(TEST[i])), list(normalize(hyps[i]))) / len(normalize(TEST[i])))
for name, i in [("best", item[0]), ("median", item[len(item) // 2]), ("worst", item[-1])]:
    c = distance(list(normalize(TEST[i])), list(normalize(hyps[i]))) / len(normalize(TEST[i]))
    print(f"{name} item (CER {pct(c)}):\n  REF {TEST[i]}\n  HYP {hyps[i]}")

In [ ]:
import subprocess, sys
# Which language tokens does this Whisper export accept? An unknown token ends the process,
# so each probe runs in a subprocess.
probe = """import sys, numpy as np, sherpa_onnx
r = sherpa_onnx.OfflineRecognizer.from_whisper(encoder=sys.argv[1], decoder=sys.argv[2], tokens=sys.argv[3], language=sys.argv[4])
s = r.create_stream(); s.accept_waveform(16000, np.zeros(16000, dtype=np.float32)); r.decode_stream(s)"""
root = archive("asr-models/sherpa-onnx-whisper-small.tar.bz2")
files = [str(root / f) for f in ("small-encoder.int8.onnx", "small-decoder.int8.onnx", "small-tokens.txt")]
ok = {l: subprocess.run([sys.executable, "-c", probe, *files, l], capture_output=True).returncode == 0
      for l in ["tr", "az", "uz", "kk", "tt", "tk", "ba", "ky", "ug", "sah", "cv"]}
print("language token accepted:", " ".join(f"{l}={'yes' if v else 'no'}" for l, v in ok.items()))

def cyrillic(text):
    letters = [c for c in text if c.isalpha()]
    return sum("CYRILLIC" in unicodedata.name(c, "") for c in letters) / max(1, len(letters))

KK = TEST[:30]
kk_waves = [speech(s, sid=i % 6) for i, s in enumerate(KK)]
print(f"Kazakh synthetic speech, {len(KK)} items (references in Cyrillic):")
print(f"  {'token':<6} {'WER':>7} {'CER':>7} {'Cyrillic':>9}  detected")
for lang in ["kk", "tt", "ba", "uz", "tr", "ru", ""]:
    res = [transcribe(w, lang) for w in kk_waves]
    e = errors(KK, [r[0] for r in res])
    det = collections.Counter(r[1] for r in res) if not lang else ""
    print(f"  {lang or 'auto':<6} {pct(e['normalized_wer'])} {pct(e['normalized_cer'])} "
          f"{statistics.mean(cyrillic(r[0]) for r in res):9.0%}  {dict(det) if det else ''}")
uz = transcribe(kk_waves[0], "uz")[0]
print(f"  item 0 with uz: {uz}\n  reference    : {KK[0]}")

TR = test_sentences("en-tr", "tr", 20)
tr_waves = [speech(s, lang="tr") for s in TR]
print(f"Turkish synthetic speech (one voice), {len(TR)} items:")
for lang in ["tr", "az", "kk", ""]:
    res = [transcribe(w, lang) for w in tr_waves]
    e = errors(TR, [r[0] for r in res])
    det = collections.Counter(r[1] for r in res) if not lang else ""
    print(f"  {lang or 'auto':<6} WER {pct(e['normalized_wer'])}  CER {pct(e['normalized_cer'])}  {dict(det) if det else ''}")

## Worked solution

Output of `experiments/ch11_exercises/run_ch11_exercises.py`:

```
language token accepted: tr=yes az=yes uz=yes kk=yes tt=yes tk=yes ba=yes ky=no ug=no sah=no cv=no
Kazakh synthetic speech, 30 items (references in Cyrillic):
  token      WER     CER  Cyrillic  detected
  kk      93.5%  54.4%      100%  
  tt     102.7%  83.1%       29%  
  ba     103.4%  91.4%        7%  
  uz     103.6%  92.4%        0%  
  tr     105.1%  94.2%        0%  
  ru     105.8%  61.0%      100%  
  auto    95.9%  70.9%       53%  {'kk': 16, 'az': 5, 'hy': 6, 'mn': 3}
  item 0 with uz: Maxsat lang ki sterge, suwarqal ienu debkosta akhparat tqag
  reference    : Мақсат - лаңкестерге су арқылы ену, деп қосты ақпараттық агенттік.
Turkish synthetic speech (one voice), 20 items:
  tr     WER  24.4%  CER   5.9%  
  az     WER  66.2%  CER  23.6%  
  kk     WER  97.1%  CER  90.3%  
  auto   WER  24.4%  CER   5.9%  {'tr': 20}
```

**Accepted tokens.** The export accepts the seven Turkic tokens the chapter lists (`tr`, `az`, `uz`, `kk`, `tt`, `tk`, `ba`) and rejects `ky`, `ug`, `sah` and `cv`; sherpa-onnx ends the process on an unknown token, which is why the probe runs in a subprocess. There is no token to force for Kyrgyz or Uyghur.

**The token decides the script and much of the accuracy.** On the same Kazakh audio, `kk` gives 54.4% CER with entirely Cyrillic output. Every other tested Turkic token gives higher CER: with `tt` the output is only 29% Cyrillic and CER rises to 83.1%; with `ba`, `uz` and `tr` the model transliterates into Latin script (0–7% Cyrillic) and CER exceeds 90%, with script mismatch contributing to the error against Cyrillic references — the Uzbek example shows recognisable Kazakh words (*Maxsat*, *ienu*, *akhparat*) in Latin letters. Russian keeps the Cyrillic script and gives 61.0% CER, closer to `kk` than any Turkic token. WER above 100% means substitutions, deletions and insertions together exceed the reference word count; insertions alone need not exceed it.

**Automatic detection is unreliable for Kazakh.** Left to detect the language, Whisper chose Kazakh for only 16 of the 30 utterances, and Azerbaijani (5), Armenian (6) or Mongolian (3) for the rest; CER rose to 70.9%. For Turkish the picture is different: all 20 utterances were detected as Turkish, and with `tr` or `auto` the CER is 5.9% — lower than the Kazakh score, but texts and synthesised voices differ, so this is not a controlled language comparison. Forcing `az` on these Turkish items gives about four times the CER (23.6%), and `kk` raises it to 90.3%.

**Consequences.** When the input language is known and supported, set its token explicitly for this evaluation, and score in the script of the references. For an unsupported language, forcing the token of a relative changes the output script and vocabulary rather than adding the language; the comparison the exercise asks for is between *fine-tuned* initialisations (a related-language checkpoint versus a multilingual one such as Omnilingual ASR or MMS), on nested training subsets with a fixed validation set.

## Your turn — extension

1. Repeat the Kazakh rows with Tatar audio (for example a Tatar Common Voice sample) to see whether `kk` or `tt` serves better.
2. For a Kyrgyz system, plan the comparison the chapter describes: fine-tune from a Kazakh-adapted Whisper and from a multilingual model (Omnilingual or MMS) on nested subsets of Kyrgyz training data, with one fixed validation set.